# TDI 101522 — TDGI (TD General Insurance) DQ Check

## Purpose
Checks data availability for all source tables and columns used in the TDI 101522 TDGI metric calculations.

## Pattern
Follows the TDW PT 101015 DQ check approach: one check per (table, column), recording which CDEs use each column.

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/GAMLConnections

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/Settings

In [ ]:
from datetime import date

# DQ results table
TABLE_NAME = SNAPSHOT_CATALOGUE_NAME + '.' + TABLE_NAME_DATA_AVA_SEG
today_date = str(date.today())

lob_id = '101522'
lob_desc = 'TDI - General Insurance'

def insertDQTable(SNAPSHOT_CATALOGUE_NAME, TABLE_NAME_DATA_AVA_SEG,
                  lob_id, cde_no, cde_desc, source, src_table_name,
                  data_element, availability_pct, today_date):
    query = ("INSERT INTO " + SNAPSHOT_CATALOGUE_NAME + "." + TABLE_NAME_DATA_AVA_SEG
             + " VALUES('" + lob_id + "','" + cde_no + "','" + cde_desc + "','"
             + source + "','" + src_table_name + "','" + data_element + "','"
             + str(availability_pct) + "','" + today_date + "')")
    spark.sql(query)
    return True

def check_column(cde_no, cde_desc, source, src_table, column):
    """Check availability of a column and insert into DQ table."""
    try:
        query = f"""
            SELECT count(1) as total,
                   count(CASE WHEN cast({column} as STRING) IS NOT NULL
                         AND trim(cast({column} as STRING)) != '' THEN 1 END) as valid
            FROM {src_table}
        """
        result = spark.sql(query).collect()[0]
        total, valid = result[0], result[1]
        pct = round(100.0 * valid / total, 2) if total > 0 else 0
        insertDQTable(SNAPSHOT_CATALOGUE_NAME, TABLE_NAME_DATA_AVA_SEG,
                      lob_id, cde_no, cde_desc, source, src_table,
                      column, pct, today_date)
        print(f"  {column}: {valid}/{total} = {pct}%")
    except Exception as e:
        print(f"  {column}: ERROR - {str(e)}")

print(f'DQ Check for {lob_id} ({lob_desc})')
print(f'Results table: {TABLE_NAME}')
print(f'Run date: {today_date}')

## Segment Data Sources

In [ ]:
# ============================================================
# Source: ra_fy_2025.joyce_tdigi_2725
# Main TDGI customer/policy table
# ============================================================
print('ra_fy_2025.joyce_tdigi_2725')
print('=' * 60)

check_column('SD1,1.7,4.1A', 'Segment/Centralized', 'Rahona',
             'ra_fy_2025.joyce_tdigi_2725', 'acc_account_no')

check_column('SD3,1.6,1.7', 'Segment/Centralized', 'Rahona',
             'ra_fy_2025.joyce_tdigi_2725', 'accpn_pholder_country_de')

In [ ]:
# ============================================================
# Source: ra_fy_2025.joyce_tdigi_2727
# TDGI customer occupation data
# ============================================================
print('ra_fy_2025.joyce_tdigi_2727')
print('=' * 60)

check_column('SD4', 'Segment', 'Rahona',
             'ra_fy_2025.joyce_tdigi_2727', 'accpn_pholder_occupation_tx')

In [ ]:
# ============================================================
# Source: ra_fy_2025.joyce_tdigi_3723
# ============================================================
print('ra_fy_2025.joyce_tdigi_3723')
print('=' * 60)

check_column('6.2B', 'Segment', 'Rahona',
             'ra_fy_2025.joyce_tdigi_3723', 'acc_account_no')

In [ ]:
# ============================================================
# Source: ra_fy_2025.joyce_tdigi_8729
# ============================================================
print('ra_fy_2025.joyce_tdigi_8729')
print('=' * 60)

check_column('4.1B', 'Segment', 'Rahona',
             'ra_fy_2025.joyce_tdigi_8729', 'acc_account_no')

In [ ]:
# ============================================================
# Source: ra_adido_2025.tdigi_legal_structure_2025
# Legal structure for SD5
# ============================================================
print('ra_adido_2025.tdigi_legal_structure_2025')
print('=' * 60)

check_column('SD5', 'Segment', 'ADIDO',
             'ra_adido_2025.tdigi_legal_structure_2025', 'Legal_Structure')

check_column('SD5', 'Segment', 'ADIDO',
             'ra_adido_2025.tdigi_legal_structure_2025', 'Accounts')

In [ ]:
# ============================================================
# Source: RA_FY24_VIEW.tdgi_active_policies
# Active policies for 6.5A
# ============================================================
print('RA_FY24_VIEW.tdgi_active_policies')
print('=' * 60)

check_column('6.5A', 'Segment', 'Rahona',
             'RA_FY24_VIEW.tdgi_active_policies', 'acc_account_no')

check_column('6.5A', 'Segment', 'Rahona',
             'RA_FY24_VIEW.tdgi_active_policies', 'tdihps_account_month_end_in')

check_column('6.5A', 'Segment', 'Rahona',
             'RA_FY24_VIEW.tdgi_active_policies', 'tdihps_asof_dt')

## Reference / Lookup Tables

In [ ]:
# ============================================================
# Source: ra_adido_2025.sanctions_country_risk_rating_2025
# Country risk ratings for 1.6, 5.2-5.8
# ============================================================
print('ra_adido_2025.sanctions_country_risk_rating_2025')
print('=' * 60)

check_column('1.6,5.2,5.3,5.4,5.5,5.6,5.7,5.8', 'Reference', 'ADIDO',
             'ra_adido_2025.sanctions_country_risk_rating_2025', 'country')

check_column('1.6,5.2,5.3,5.4,5.5,5.6,5.7,5.8', 'Reference', 'ADIDO',
             'ra_adido_2025.sanctions_country_risk_rating_2025', 'riskrating')

In [ ]:
# ============================================================
# Source: ra_adido_2025.occupation_code_list_Ca2025
# Occupation risk ratings for SD4
# ============================================================
print('ra_adido_2025.occupation_code_list_Ca2025')
print('=' * 60)

check_column('SD4', 'Reference', 'ADIDO',
             'ra_adido_2025.occupation_code_list_Ca2025', 'Occupation_Name')

check_column('SD4', 'Reference', 'ADIDO',
             'ra_adido_2025.occupation_code_list_Ca2025', 'Updated_Risk_Rating')

In [ ]:
# ============================================================
# Source: ra_adido_2025.industry_ref_list_ca2025
# Industry risk ratings for SD4
# ============================================================
print('ra_adido_2025.industry_ref_list_ca2025')
print('=' * 60)

check_column('SD4', 'Reference', 'ADIDO',
             'ra_adido_2025.industry_ref_list_ca2025', 'Industry_Code')

check_column('SD4', 'Reference', 'ADIDO',
             'ra_adido_2025.industry_ref_list_ca2025', 'Industry_Description')

check_column('SD4', 'Reference', 'ADIDO',
             'ra_adido_2025.industry_ref_list_ca2025', 'Updated_Risk_Rating')

In [ ]:
# ============================================================
# Source: ra_adido_2025.entity_ref_list_ca2025
# Entity type risk ratings for SD5
# ============================================================
print('ra_adido_2025.entity_ref_list_ca2025')
print('=' * 60)

check_column('SD5', 'Reference', 'ADIDO',
             'ra_adido_2025.entity_ref_list_ca2025', 'Entity_Type_Name')

check_column('SD5', 'Reference', 'ADIDO',
             'ra_adido_2025.entity_ref_list_ca2025', 'Entity_Category')

check_column('SD5', 'Reference', 'ADIDO',
             'ra_adido_2025.entity_ref_list_ca2025', 'Updated_Risk_Ratings')

## Branch Data

In [ ]:
# ============================================================
# Source: RA_FY25_VIEW.TD_Insurance_Branches_2025
# Branch data for 5.1-5.9
# ============================================================
print('RA_FY25_VIEW.TD_Insurance_Branches_2025')
print('=' * 60)

check_column('5.1,5.9', 'Branch', 'ADIDO',
             'RA_FY25_VIEW.TD_Insurance_Branches_2025', 'general_ins_ind')

check_column('5.2,5.3,5.4,5.5,5.6,5.7,5.8', 'Branch', 'ADIDO',
             'RA_FY25_VIEW.TD_Insurance_Branches_2025', 'country')

## Policy History Data (1.7)

In [ ]:
# ============================================================
# Source: ra_fy_2025.prod30tdi_tddi_to_active_policies_history
# Policy history for metric 1.7
# ============================================================
print('ra_fy_2025.prod30tdi_tddi_to_active_policies_history')
print('=' * 60)

check_column('1.7', 'Centralized', 'Rahona',
             'ra_fy_2025.prod30tdi_tddi_to_active_policies_history', 'acc_account_no')

check_column('1.7', 'Centralized', 'Rahona',
             'ra_fy_2025.prod30tdi_tddi_to_active_policies_history', 'pol_valid_la')

check_column('1.7', 'Centralized', 'Rahona',
             'ra_fy_2025.prod30tdi_tddi_to_active_policies_history', 'pol_effective_dt')

In [ ]:
# ============================================================
# Source: ra_fy_2025.prod30tdi_tdi_account_person
# Account-person mapping for 1.7
# ============================================================
print('ra_fy_2025.prod30tdi_tdi_account_person')
print('=' * 60)

check_column('1.7', 'Centralized', 'Rahona',
             'ra_fy_2025.prod30tdi_tdi_account_person', 'acc_account_no')

check_column('1.7', 'Centralized', 'Rahona',
             'ra_fy_2025.prod30tdi_tdi_account_person', 'accpn_pholder_country_de')

## Centralized / Sanctions / STR / UTR

In [ ]:
# ============================================================
# Source: ra_adido_2025.true_sanctions_match_2025
# True sanctions matching for 1.8
# ============================================================
print('ra_adido_2025.true_sanctions_match_2025')
print('=' * 60)

check_column('1.8', 'Centralized', 'ADIDO',
             'ra_adido_2025.true_sanctions_match_2025', 'Customername')

check_column('1.8', 'Centralized', 'ADIDO',
             'ra_adido_2025.true_sanctions_match_2025', 'tds')

In [ ]:
# ============================================================
# Source: ra_adido_2025.customer_sanctioned_blocked_property_2025
# Blocked property for 1.9
# ============================================================
print('ra_adido_2025.customer_sanctioned_blocked_property_2025')
print('=' * 60)

check_column('1.9', 'Centralized', 'ADIDO',
             'ra_adido_2025.customer_sanctioned_blocked_property_2025', 'ACCOUNTBLOCKED')

check_column('1.9', 'Centralized', 'ADIDO',
             'ra_adido_2025.customer_sanctioned_blocked_property_2025', 'CUSTOMERIDIMPACTED')

In [ ]:
# ============================================================
# Source: rafy2025_centralized.crv_scorable_cust_ca
# CRV scoring for 1.1 (unscored customers)
# ============================================================
print('rafy2025_centralized.crv_scorable_cust_ca')
print('=' * 60)

check_column('1.1', 'Centralized', 'Rahona',
             'rafy2025_centralized.crv_scorable_cust_ca', 'CUST_INTRL_ID')

check_column('1.1', 'Centralized', 'Rahona',
             'rafy2025_centralized.crv_scorable_cust_ca', 'v_entity_id')

In [ ]:
# ============================================================
# Source: rafy2025_centralized.customer_scorable_rated_ca
# Risk-rated customers for 1.2, 1.3, 1.4
# ============================================================
print('rafy2025_centralized.customer_scorable_rated_ca')
print('=' * 60)

check_column('1.2,1.3,1.4', 'Centralized', 'Rahona',
             'rafy2025_centralized.customer_scorable_rated_ca', 'CUST_INTRL_ID')

check_column('1.2,1.3,1.4', 'Centralized', 'Rahona',
             'rafy2025_centralized.customer_scorable_rated_ca', 'risk_rating')

In [ ]:
# ============================================================
# Source: rafy2025_centralized.customer_scorable_unrated_ca
# Unrated customers for 1.5
# ============================================================
print('rafy2025_centralized.customer_scorable_unrated_ca')
print('=' * 60)

check_column('1.5', 'Centralized', 'Rahona',
             'rafy2025_centralized.customer_scorable_unrated_ca', 'CUST_INTRL_ID')

In [ ]:
# ============================================================
# Source: rafy2025_centralized.CDE_SD_6_1yr_fy2025
# New customers (<1yr) for SD6
# ============================================================
print('rafy2025_centralized.CDE_SD_6_1yr_fy2025')
print('=' * 60)

check_column('SD6', 'Centralized', 'Rahona',
             'rafy2025_centralized.CDE_SD_6_1yr_fy2025', 'full_nm')

In [ ]:
# ============================================================
# Source: rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details
# Unusual Transaction Referrals for 3.17
# ============================================================
print('rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details')
print('=' * 60)

check_column('3.17', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'full_nm')

check_column('3.17', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'cust_no')

check_column('3.17', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'birth_dt')

In [ ]:
# ============================================================
# Source: rafy2025_centralized.TD_SAR_CDE_3_18_2025
# SARs/STRs for 3.18
# ============================================================
print('rafy2025_centralized.TD_SAR_CDE_3_18_2025')
print('=' * 60)

check_column('3.18', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_SAR_CDE_3_18_2025', 'STR_Internal_Unique_ID')

check_column('3.18', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_SAR_CDE_3_18_2025', 'STR_LEILAs_Customer_Nam')

## ABAC Data Sources

In [ ]:
# ============================================================
# Source: ra_adido_2025.pep_list_2025_exploded
# PEP list for ABAC1.2
# ============================================================
print('ra_adido_2025.pep_list_2025_exploded')
print('=' * 60)

check_column('ABAC1.2', 'ABAC', 'ADIDO',
             'ra_adido_2025.pep_list_2025_exploded', 'ENTITY')

check_column('ABAC1.2', 'ABAC', 'ADIDO',
             'ra_adido_2025.pep_list_2025_exploded', 'CUST_INTRL_ID')

In [ ]:
# ============================================================
# Source: ra_adido_2025.TD_Country_Risk_Rating_ABAC
# ABAC country risk for ABAC5.1
# ============================================================
print('ra_adido_2025.TD_Country_Risk_Rating_ABAC')
print('=' * 60)

check_column('ABAC5.1', 'ABAC', 'ADIDO',
             'ra_adido_2025.TD_Country_Risk_Rating_ABAC', 'DerivedDesc')

check_column('ABAC5.1', 'ABAC', 'ADIDO',
             'ra_adido_2025.TD_Country_Risk_Rating_ABAC', 'FinalABACRating')

## Summary

In [ ]:
# ============================================================
# Display all DQ results for this AU
# ============================================================
results = spark.sql(f"""
    SELECT * FROM {TABLE_NAME}
    WHERE lob_id = '{lob_id}'
      AND run_date = '{today_date}'
    ORDER BY src_table_name, data_element
""")

print(f'Total DQ checks: {results.count()}')
display(results)